# [9.1] Refusal Directions and Safe Steering

> **ARENA extension note.** Original ARENA content is unchanged. This appended alignment section keeps prompt content sanitized and teaches the refusal-direction mechanics on visible toy activations before checking a pinned CUDA report.

By the end of this notebook, you will have shown that a mean-difference residual-stream direction can separate refusal-labeled and harmless prompts in a controlled activation stack, and that addition/projection interventions change refusal evidence beyond random-direction and label-shuffled controls.

## Core Question

When someone says a model has a "refusal direction", what evidence would make that claim hard to fake?

A scalar direction is only interesting if it separates held-out examples, appears at plausible layers/positions, changes behavior under intervention, fails under random and label-shuffled controls, and does not simply wreck benign capability.

**Common bug:** treating a classifier or pretty projection plot as the whole result. In this notebook, the claim only passes when the same direction survives held-out tests, steering/projection checks, and negative controls.

## Learning Objectives

By the end, you should be able to:

1. build safe prompt categories without storing harmful procedural text;
2. implement a normalized mean-difference direction;
3. project activations onto that direction and score held-out separation;
4. run a layer sweep and interpret where the direction emerges;
5. compute a small PCA/SVD geometry check;
6. test addition and projection-out curves against random-direction controls;
7. reject label-shuffled shortcuts and broad capability collapse; and
8. read the GT-2 CUDA report as supporting evidence rather than the lesson itself.


In [ ]:
GT_TIER = "GT-2"
EXERCISE_ID = "9_1_refusal_directions_and_safe_steering"
DIFFICULTY = 4
IMPORTANCE = 4
EXPECTED_RUNTIME = "45-60 minutes for exercises; minutes for CUDA real-model report regeneration"
REQUIRES_GPU = True  # exercises are CPU-friendly; full acceptance uses pinned CUDA preflights


## Setup

The toy activations below are deliberately small, but they are not arbitrary one-off tensors. They form a six-layer activation stack with train and held-out examples, nuisance dimensions, and controls. This lets you do the same reasoning you would use with a real residual stream while keeping the learner path fast and inspectable.


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch as t
from IPython.display import display

chapter = "chapter9_alignment_interpretability"
section = "part1_refusal_directions_safe_steering"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_refusal_directions_safe_steering.tests as tests


### Exercise - Build Safe Prompt Pairs and Toy Activations

> ```yaml
> Difficulty: easy
> Importance: high
> ```
>
> You should spend 10 minutes on this exercise.

Create a balanced set of sanitized refusal-labeled and allowed prompt categories. Do not write harmful procedural content; the table should contain only safe summaries. The activation stack should have shape `(layers, examples, d_model)` and a train/held-out split.


In [ ]:
def toy_refusal_activation_batch() -> dict:
    raise NotImplementedError()


batch = toy_refusal_activation_batch()
display(pd.DataFrame(batch["prompt_table"]))
print({
    "activation_shape": tuple(batch["activations_by_layer"].shape),
    "train_examples": int(batch["train_mask"].sum().item()),
    "heldout_examples": int((~batch["train_mask"]).sum().item()),
})
assert batch["activations_by_layer"].shape == (6, 12, 4)
assert int(batch["labels"].sum().item()) == 6


<details>
<summary>Expected output</summary>

You should see 12 safe prompt-category rows, 6 train examples, 6 held-out examples, and activation shape `(6, 12, 4)`.

</details>

<details>
<summary>Help - what makes this safe?</summary>

The table names categories, not harmful instructions. The real GT-2 report follows the same principle: it stores aggregate metrics and prompt hashes, not raw prompts or generated completion text.

</details>

<details>
<summary>Common bugs</summary>

- Letting prompt text leak into committed artifacts.
- Mixing train and held-out examples before evaluating the direction.
- Building a one-layer fixture, which prevents layer-sweep reasoning.

</details>

<details>
<summary>Solution</summary>

Construct balanced refusal/allowed labels, a boolean train mask, and a deterministic layer stack where the refusal axis strengthens over layers while nuisance dimensions fade.

</details>


### Exercise - Mean-Difference Direction and Projection Scores

> ```yaml
> Difficulty: easy
> Importance: high
> ```
>
> You should spend 10 minutes on this exercise.

Implement the core operation yourself. A refusal direction points from the mean allowed activation to the mean refusal activation, then gets normalized. Projection scores are signed dot products along that direction.


In [ ]:
def mean_difference_direction(refusal_activations: t.Tensor, allowed_activations: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


def refusal_direction_scores(activations: t.Tensor, direction: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


def direction_smoke_test() -> list[float]:
    refusal = t.tensor([[2.0, 0.0], [2.0, 1.0]])
    non_refusal = t.tensor([[0.0, 0.0], [0.0, 1.0]])
    return mean_difference_direction(refusal, non_refusal).tolist()


def scores_smoke_test() -> list[float]:
    activations = t.tensor([[2.0, 0.0], [0.5, 1.0]])
    direction = t.tensor([1.0, 0.0])
    return refusal_direction_scores(activations, direction).tolist()


tests.test_direction_smoke_test(direction_smoke_test)
tests.test_scores_smoke_test(scores_smoke_test)


<details>
<summary>Expected output</summary>

```text
All tests in `test_direction_smoke_test` passed!
All tests in `test_scores_smoke_test` passed!
```

</details>

<details>
<summary>Help - check the sign convention</summary>

Refusal examples should score higher than allowed examples. If the margin is negative later, you probably subtracted in the wrong order.

</details>

<details>
<summary>Common bugs</summary>

- Forgetting to normalize the direction.
- Returning scores with shape `(examples, 1)` instead of `(examples,)`.
- Normalizing activations instead of the direction.

</details>

<details>
<summary>Solution</summary>

Subtract class means as `refusal_mean - allowed_mean`, divide by the vector norm, then compute `activations @ unit_direction`.

</details>


### Exercise - Held-Out Separation

> ```yaml
> Difficulty: medium
> Importance: high
> ```
>
> You should spend 10 minutes on this exercise.

A direction is not evidence until it separates held-out examples. Implement the report using accuracy and signed margin.


In [ ]:
def refusal_separation_report(
    activations: t.Tensor,
    labels: t.Tensor,
    direction: t.Tensor,
    *,
    min_accuracy: float = 0.9,
) -> dict:
    raise NotImplementedError()


def separation_smoke_test() -> dict:
    activations = t.tensor([[2.0, 0.0], [3.0, 0.0], [0.0, 0.0], [0.5, 0.0]])
    labels = t.tensor([1, 1, 0, 0], dtype=t.bool)
    direction = t.tensor([1.0, 0.0])
    return refusal_separation_report(activations, labels, direction, min_accuracy=0.9)


tests.test_separation_smoke_test(separation_smoke_test)


<details>
<summary>Expected output</summary>

```text
All tests in `test_separation_smoke_test` passed!
```

The toy margin should be `2.25` and the accuracy should be `1.0`.

</details>

<details>
<summary>Help - why margin and accuracy?</summary>

Accuracy can be high with a tiny separation. Margin tells you whether the refusal and allowed score distributions are actually separated.

</details>

<details>
<summary>Common bugs</summary>

- Evaluating on train examples only.
- Using a fixed threshold before checking score orientation.
- Treating negative margin as acceptable.

</details>

<details>
<summary>Solution</summary>

Compute scores, choose the midpoint between refusal and allowed means as a threshold, then report accuracy plus `refusal_mean - allowed_mean`.

</details>


### Exercise - Layer Sweep

> ```yaml
> Difficulty: medium
> Importance: high
> ```
>
> You should spend 15 minutes on this exercise.

Do not pick a layer because the report said so. Recompute the direction at every toy layer, evaluate on held-out examples, and plot accuracy and margin.


In [ ]:
def layer_sweep(batch: dict) -> list[dict]:
    raise NotImplementedError()


sweep_rows = layer_sweep(batch)
display(pd.DataFrame(sweep_rows))
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].plot([row["layer"] for row in sweep_rows], [row["heldout_accuracy"] for row in sweep_rows], marker="o")
axes[0].set_title("Held-out accuracy by layer")
axes[0].set_xlabel("layer")
axes[0].set_ylim(0, 1.05)
axes[1].plot([row["layer"] for row in sweep_rows], [row["heldout_margin"] for row in sweep_rows], marker="o", color="tab:green")
axes[1].set_title("Held-out margin by layer")
axes[1].set_xlabel("layer")
plt.tight_layout()
plt.show()
assert sweep_rows[-1]["heldout_margin"] > sweep_rows[0]["heldout_margin"]


<details>
<summary>Expected output</summary>

The first layer should be weak or wrong, while later layers should reach held-out accuracy `1.0`; the margin should climb to above `4.0` by the final toy layer.

</details>

<details>
<summary>Help - interpreting the layer curve</summary>

The curve is evidence about where this toy stack linearly separates the categories. It is not evidence that every model has a final-layer refusal feature.

</details>

<details>
<summary>Common bugs</summary>

- Fitting the direction on all examples and then calling it held-out.
- Reusing one direction across all layers instead of recomputing it per layer.
- Plotting only accuracy and hiding a weak margin.

</details>

<details>
<summary>Solution</summary>

Loop over layers, fit on train examples only, evaluate on held-out examples only, and store both accuracy and margin.

</details>


### Exercise - PCA/SVD Geometry Check

> ```yaml
> Difficulty: medium
> Importance: medium
> ```
>
> You should spend 10 minutes on this exercise.

A mean-difference direction is one vector. PCA/SVD asks whether refusal-vs-allowed differences have a dominant direction or are spread across many directions.


In [ ]:
def pca_svd_refusal_geometry(batch: dict, *, layer: int = -1) -> dict:
    raise NotImplementedError()


geometry = pca_svd_refusal_geometry(batch)
print(geometry)
fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(range(len(geometry["variance_fractions"])), geometry["variance_fractions"])
ax.set_title("Variance fractions of paired refusal-minus-allowed differences")
ax.set_xlabel("component")
ax.set_ylabel("fraction")
plt.show()
assert geometry["pc1_variance_fraction"] > 0.5


<details>
<summary>Expected output</summary>

The first component should explain most toy paired-difference variance, usually above `0.8` for this fixture.

</details>

<details>
<summary>Help - what PCA does and does not prove</summary>

A high PC1 fraction supports the idea that one dominant contrast exists in this toy stack. It does not prove the direction is causal; that comes from intervention and controls.

</details>

<details>
<summary>Common bugs</summary>

- Running PCA on raw activations instead of refusal-minus-allowed differences.
- Using held-out examples to pick the direction.
- Reporting a 2D plot without variance or control numbers.

</details>

<details>
<summary>Solution</summary>

Pair train refusal and allowed activations, subtract them, center the differences, run `torch.linalg.svdvals`, and normalize squared singular values.

</details>


### Exercise - Addition and Projection-Out Curves

> ```yaml
> Difficulty: hard
> Importance: high
> ```
>
> You should spend 15 minutes on this exercise.

Addition asks whether moving allowed activations along the refusal direction increases refusal evidence. Projection-out asks whether removing the direction from refusal activations lowers refusal evidence.


In [ ]:
def steering_and_projection_curves(batch: dict, *, layer: int = -1) -> dict:
    raise NotImplementedError()


curves = steering_and_projection_curves(batch)
addition_df = pd.DataFrame(curves["addition_curve"])
projection_df = pd.DataFrame(curves["projection_curve"])
display(addition_df)
display(projection_df)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(addition_df["alpha"], addition_df["target_mean_score"], marker="o", label="target direction")
axes[0].plot(addition_df["alpha"], addition_df["random_mean_score"], marker="o", label="random direction")
axes[0].set_title("Addition on allowed examples")
axes[0].set_xlabel("alpha")
axes[0].set_ylabel("mean refusal score")
axes[0].legend()
axes[1].plot(projection_df["projection_fraction"], projection_df["target_mean_score"], marker="o", label="project target")
axes[1].plot(projection_df["projection_fraction"], projection_df["random_mean_score"], marker="o", label="project random")
axes[1].set_title("Projection-out on refusal examples")
axes[1].set_xlabel("projection fraction")
axes[1].legend()
plt.tight_layout()
plt.show()
assert addition_df.iloc[-1]["target_mean_score"] > addition_df.iloc[-1]["random_mean_score"] + 1.0
assert projection_df.iloc[-1]["target_mean_score"] < projection_df.iloc[-1]["random_mean_score"] - 1.0


<details>
<summary>Expected output</summary>

The addition curve should rise for the target direction but stay almost flat for the random direction. The projection-out curve should lower target refusal scores while random projection barely moves them.

</details>

<details>
<summary>Help - addition versus projection-out</summary>

Addition and projection-out are different causal tests. Addition on allowed prompts asks whether the direction can induce refusal evidence; projection-out on refusal prompts asks whether removing it erases evidence.

</details>

<details>
<summary>Common bugs</summary>

- Comparing target addition to no baseline instead of a random direction.
- Projecting out the wrong vector or forgetting to normalize it.
- Treating a classifier score change as behavioral proof without limitations.

</details>

<details>
<summary>Solution</summary>

Use held-out allowed scores for addition, held-out refusal scores for projection-out, and compute matched random-direction effects in the same score space.

</details>


### Exercise - Controls and Side Effects

> ```yaml
> Difficulty: medium
> Importance: high
> ```
>
> You should spend 15 minutes on this exercise.

A refusal-direction claim should fail under label shuffling and random directions. It should also bound benign capability degradation.


In [ ]:
def capability_degradation_report(baseline_scores: t.Tensor, steered_scores: t.Tensor, *, max_degradation: float = 0.1) -> dict:
    raise NotImplementedError()


def random_direction_control_report(*, target_direction_delta: float, random_direction_delta: float, min_margin: float = 0.2) -> dict:
    raise NotImplementedError()


def label_shuffle_control_report(activations: t.Tensor, labels: t.Tensor, *, min_accuracy_gap: float = 0.25) -> dict:
    raise NotImplementedError()


def capability_smoke_test() -> dict:
    return capability_degradation_report(t.tensor([0.9, 0.8]), t.tensor([0.85, 0.75]), max_degradation=0.1)


def random_control_smoke_test() -> dict:
    return random_direction_control_report(target_direction_delta=0.4, random_direction_delta=0.05, min_margin=0.2)


def label_shuffle_smoke_test() -> dict:
    activations = t.tensor([[3.0, 0.0], [2.5, 0.0], [0.0, 0.0], [0.2, 0.0]])
    labels = t.tensor([1, 1, 0, 0], dtype=t.bool)
    return label_shuffle_control_report(activations, labels, min_accuracy_gap=0.25)


tests.test_capability_smoke_test(capability_smoke_test)
tests.test_random_control_smoke_test(random_control_smoke_test)
tests.test_label_shuffle_smoke_test(label_shuffle_smoke_test)


<details>
<summary>Expected output</summary>

```text
All tests in `test_capability_smoke_test` passed!
All tests in `test_random_control_smoke_test` passed!
All tests in `test_label_shuffle_smoke_test` passed!
```

</details>

<details>
<summary>Help - why label shuffle?</summary>

If a direction works when labels are deliberately mismatched, it is probably detecting nuisance structure or leakage rather than refusal behavior.

</details>

<details>
<summary>Common bugs</summary>

- Using the same random direction each time without a declared seed or fixture.
- Comparing shuffled-label accuracy to zero rather than to the true-label result.
- Calling steering safe without any side-effect measurement.

</details>

<details>
<summary>Solution</summary>

Compute target-vs-random margins in the same units, build a maximally mismatched label assignment, and report baseline-minus-steered capability degradation.

</details>


### Exercise - Candidate Comparison and Signature Result

> ```yaml
> Difficulty: medium
> Importance: high
> ```
>
> You should spend 10 minutes on this exercise.

Collect the visible evidence into a single signature result. This is the lesson; the committed CUDA report is the appendix evidence.


In [ ]:
def direction_comparison_report(method_scores: dict[str, float]) -> dict:
    raise NotImplementedError()


def comparison_smoke_test() -> dict:
    return direction_comparison_report({
        "mean_difference": 0.95,
        "probe": 0.9,
        "sae_feature": 0.85,
        "gemma_scope": 0.8,
    })


def toy_refusal_signature_result() -> dict:
    raise NotImplementedError()


tests.test_comparison_smoke_test(comparison_smoke_test)
tests.test_toy_refusal_signature_result_has_visible_curves(toy_refusal_signature_result)


<details>
<summary>Expected output</summary>

```text
All tests in `test_comparison_smoke_test` passed!
All tests in `test_toy_refusal_signature_result_has_visible_curves` passed!
```

</details>

<details>
<summary>Help - what counts as the result?</summary>

The result is not one scalar. It is the bundle: visible prompt categories, layer sweep, geometry check, addition/projection curves, failed controls, and bounded side effects.

</details>

<details>
<summary>Common bugs</summary>

- Declaring victory from held-out accuracy alone.
- Treating the verification report as the main notebook output.
- Hiding the control failures in prose instead of plotting or tabulating them.

</details>

<details>
<summary>Solution</summary>

Assemble the outputs from the earlier exercises, then require every control to pass before setting `control_claim_passed`.

</details>


## Signature Result

This cell generates the learner-facing result: a prompt-category table, a layer sweep, a PCA/SVD geometry check, intervention curves, and a control table. If any control fails, the correct interpretation is a negative result.


In [ ]:
signature = toy_refusal_signature_result()

summary_df = pd.DataFrame([
    {"check": "held-out accuracy", "value": signature["heldout_accuracy"], "passes": signature["heldout_accuracy"] == 1.0},
    {"check": "held-out margin", "value": signature["heldout_margin"], "passes": signature["heldout_margin"] > 3.0},
    {"check": "PC1 variance fraction", "value": signature["pc1_variance_fraction"], "passes": signature["pc1_variance_fraction"] > 0.5},
    {"check": "label-shuffle control", "value": signature["label_shuffle_shuffled_accuracy"], "passes": signature["label_shuffle_fails"]},
    {"check": "random-direction control", "value": signature["random_direction_margin"], "passes": signature["random_direction_fails"]},
    {"check": "capability degradation", "value": signature["capability_degradation"], "passes": signature["capability_degradation_small"]},
])
display(summary_df)

sweep_df = pd.DataFrame(signature["layer_sweep"])
addition_df = pd.DataFrame(signature["addition_curve"])
projection_df = pd.DataFrame(signature["projection_curve"])

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
axes[0, 0].plot(sweep_df["layer"], sweep_df["heldout_accuracy"], marker="o", label="accuracy")
axes[0, 0].set_ylim(0, 1.05)
axes[0, 0].set_title("Layer sweep: held-out accuracy")
axes[0, 1].plot(sweep_df["layer"], sweep_df["heldout_margin"], marker="o", color="tab:green")
axes[0, 1].set_title("Layer sweep: signed margin")
axes[1, 0].plot(addition_df["alpha"], addition_df["target_mean_score"], marker="o", label="target")
axes[1, 0].plot(addition_df["alpha"], addition_df["random_mean_score"], marker="o", label="random")
axes[1, 0].set_title("Addition on allowed examples")
axes[1, 0].set_xlabel("alpha")
axes[1, 0].legend()
axes[1, 1].plot(projection_df["projection_fraction"], projection_df["target_mean_score"], marker="o", label="target")
axes[1, 1].plot(projection_df["projection_fraction"], projection_df["random_mean_score"], marker="o", label="random")
axes[1, 1].set_title("Projection-out on refusal examples")
axes[1, 1].set_xlabel("projection fraction")
axes[1, 1].legend()
plt.tight_layout()
plt.show()

assert signature["control_claim_passed"]


## Try It Yourself

Change the prompt, alpha, or layer below. The toy function maps benign prompts to held-out allowed examples; it does not generate text or display harmful content.


In [ ]:
def run_with_refusal_direction(prompt: str, alpha: float, *, layer: int = 5) -> pd.DataFrame:
    batch = toy_refusal_activation_batch()
    labels = batch["labels"]
    train_mask = batch["train_mask"]
    activations = batch["activations_by_layer"][layer]
    direction = mean_difference_direction(activations[train_mask & labels], activations[train_mask & ~labels])
    prompt_index = 10 if "birthday" in prompt.lower() else 9
    baseline_score = refusal_direction_scores(activations[prompt_index : prompt_index + 1], direction)[0].item()
    steered_score = baseline_score + alpha
    return pd.DataFrame([
        {"prompt": prompt, "layer": layer, "alpha": 0.0, "refusal_score": baseline_score},
        {"prompt": prompt, "layer": layer, "alpha": alpha, "refusal_score": steered_score},
    ])


def plot_refusal_projection_distribution(layer: int = 5) -> None:
    batch = toy_refusal_activation_batch()
    labels = batch["labels"]
    train_mask = batch["train_mask"]
    activations = batch["activations_by_layer"][layer]
    direction = mean_difference_direction(activations[train_mask & labels], activations[train_mask & ~labels])
    scores = refusal_direction_scores(activations[~train_mask], direction)
    heldout_labels = labels[~train_mask]
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.scatter(scores[heldout_labels], [1] * int(heldout_labels.sum()), label="refusal", s=60)
    ax.scatter(scores[~heldout_labels], [0] * int((~heldout_labels).sum()), label="allowed", s=60)
    ax.set_yticks([0, 1], ["allowed", "refusal"])
    ax.set_xlabel("projection score")
    ax.set_title(f"Held-out projection distribution at layer {layer}")
    ax.legend()
    plt.show()


prompt = "Write a polite birthday message to my friend."
alpha = 4.0
display(run_with_refusal_direction(prompt, alpha, layer=5))
plot_refusal_projection_distribution(layer=5)


## Notebook Contract

The CPU smoke contract now includes the visible toy signature result. The CUDA report below remains the scoped GT-2 evidence path.


In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    return {
        "toy_signature": toy_refusal_signature_result(),
        "direction": direction_smoke_test(),
        "scores": scores_smoke_test(),
        "separation": separation_smoke_test(),
        "steering": steering_smoke_test(),
        "capability": capability_smoke_test(),
        "random_control": random_control_smoke_test(),
        "label_shuffle": label_shuffle_smoke_test(),
        "comparison": comparison_smoke_test(),
    }


def steering_smoke_test() -> dict:
    baseline = t.tensor([0.2, 0.4, 0.7])
    steered = t.tensor([0.8, 0.9, 0.4])
    baseline_rate = baseline.ge(0.5).float().mean().item()
    steered_rate = steered.ge(0.5).float().mean().item()
    return {
        "baseline_refusal_rate": baseline_rate,
        "steered_refusal_rate": steered_rate,
        "refusal_rate_delta": steered_rate - baseline_rate,
        "changes_refusal_rate": steered_rate - baseline_rate >= 0.3,
    }


tests.test_steering_smoke_test(steering_smoke_test)
tests.test_notebook_contract(run_smoke_test)


## CUDA Report as Supporting Evidence

The committed GT-2 report uses pinned Qwen2.5-0.5B-Instruct and the public `josephmayo/refusal-compliance-pairs` dataset. It stores aggregate metrics and prompt hashes only. This is review evidence, not the main learner-facing result.


In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
report_summary = pd.DataFrame([
    {"metric": "GT-2 held-out accuracy", "value": gpu["gt2_refusal_direction_heldout_accuracy"]},
    {"metric": "GT-2 held-out margin", "value": gpu["gt2_refusal_direction_heldout_margin"]},
    {"metric": "best layer", "value": gpu["gt2_refusal_direction_layer_sweep_best_layer"]},
    {"metric": "PC1 variance fraction", "value": gpu["gt2_refusal_direction_pc1_variance_fraction"]},
    {"metric": "random direction fails", "value": gpu["gt2_refusal_direction_random_direction_fails"]},
    {"metric": "label shuffle fails", "value": gpu["gt2_refusal_direction_label_shuffle_fails"]},
    {"metric": "raw prompt text saved", "value": gpu["gt2_refusal_direction_raw_prompt_text_saved"]},
    {"metric": "completion text saved", "value": gpu["gt2_refusal_direction_completion_text_saved"]},
    {"metric": "peak VRAM GB", "value": round(gpu["peak_vram_gb"], 3)},
])
display(report_summary)


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    from part1_refusal_directions_safe_steering import solutions as reference_solutions
    return reference_solutions.run_gpu_test(max_vram_gb=max_vram_gb)


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    from part1_refusal_directions_safe_steering import solutions as reference_solutions
    return reference_solutions.run_full_experiment(max_vram_gb=max_vram_gb)


tests.test_committed_gpu_report_matches_refusal_direction_contract(report)


## Limitations and Anomaly Hunting

The toy activations prove the mechanics, not a claim about a deployed model. The CUDA report supports a scoped GT-2 aggregate replication: it uses pinned models, public data, sanitized handling, prompt hashes, and aggregate metrics. It does not save raw harmful prompts or generated completion text.

Treat the following as negative results, not polish targets:

- the target direction fails to beat random directions;
- label shuffling preserves the result;
- projection plots look good but held-out accuracy or margin is weak;
- addition/projection changes only by collapsing benign capability;
- a behavioral diagnostic fails its own random-control comparison.

## Reading Links

- ArXiv 2406.11717, "Refusal is Mediated by a Single Direction".
- Original ARENA function-vector and steering material in Chapter 1.3.2.
- The local `verification_report.json` for exact model revisions, dataset revision, aggregate metrics, and VRAM.
